In [29]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import re
import warnings

# 忽略 Pandas 和 Matplotlib 的一些版本警告
warnings.filterwarnings("ignore")

# --- Matplotlib 中文設定 ---
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei'] 
plt.rcParams['axes.unicode_minus'] = False

# %%
# 1. 參數設定區
# ==========================================
# 特徵檔案路徑 (請根據實際情況修改)
BASE_DIR = r"C:\Users\kein9\OneDrive\桌面\LAB\RelaxingBC_AC\FeatureBC&AC\BC"

# 報告輸出路徑
OUTPUT_DIR = r"C:\Users\kein9\OneDrive\桌面\LAB\RelaxingBC_AC\AnalyzeData"

# 分析目標
TARGET_FILES = ['EEG_t', 'GSR', 'HRV', 'RES_dc']
KEY_COL = 'data_name'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'Final_Report_With_Perfect_List.xlsx')

# 🔥 設定要列出幾筆資料 (預設 Top 5)
TOP_N_SHOW = 5 

# 確保輸出資料夾存在
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)


In [30]:
# 2. 核心演算法
# ==========================================
def calculate_smape(y_true, y_pred):
    """
    計算對稱平均絕對百分比誤差 (SMAPE)
    """
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred)
    # 避免 0/0
    smape = np.divide(diff, denominator, out=np.zeros_like(diff), where=denominator!=0)
    return smape

def analyze_file(base_dir, prefix):
    """
    分析函數：回傳 Top N Best 和 Top N Worst
    """
    path_senior = os.path.join(base_dir, f"{prefix}.csv")
    path_mine = os.path.join(base_dir, f"{prefix}_test.csv")
    
    if not os.path.exists(path_senior) or not os.path.exists(path_mine):
        print(f"❌ {prefix}: 找不到檔案")
        return None, [], [], None

    try:
        # 讀檔與清洗
        df_senior = pd.read_csv(path_senior)
        df_mine = pd.read_csv(path_mine)
        
        # 統一 ID 格式
        df_senior[KEY_COL] = df_senior[KEY_COL].astype(str).str.strip()
        df_mine[KEY_COL] = df_mine[KEY_COL].astype(str).str.strip().str.replace(r'_test$', '', regex=True)
        
        # 移除 Unnamed
        if 'Unnamed: 0' in df_senior.columns: df_senior.drop(columns=['Unnamed: 0'], inplace=True)
        if 'Unnamed: 0' in df_mine.columns: df_mine.drop(columns=['Unnamed: 0'], inplace=True)
        
        # 合併
        df = pd.merge(df_mine, df_senior, on=KEY_COL, suffixes=('_Mine', '_Senior'), how='inner')
        
        if df.empty: return None, [], [], None

        # 找出數值欄位
        common_cols = [c for c in df_mine.columns if c in df_senior.columns and c != KEY_COL]
        num_cols = [c for c in common_cols if pd.api.types.is_numeric_dtype(df_mine[c])]
        
        # ---------------------------------------------------------
        # 計算誤差矩陣
        # ---------------------------------------------------------
        smape_matrix = df[[KEY_COL]].copy()
        
        for col in num_cols:
            v_mine = df[f"{col}_Mine"].fillna(0)
            v_senior = df[f"{col}_Senior"].fillna(0)
            smape_matrix[col] = calculate_smape(v_senior, v_mine)

        # 統一基準：計算每個病人的 Mean SMAPE
        smape_matrix['Mean_SMAPE'] = smape_matrix[num_cols].mean(axis=1)

        # ---------------------------------------------------------
        # 1. 產生 Best List (Perfect 優先，否則取 Top N Best)
        # ---------------------------------------------------------
        perfect_mask = smape_matrix['Mean_SMAPE'] < 1e-9
        perfect_df = smape_matrix[perfect_mask]
        
        best_rows_list = []
        list_type = "Perfect" # 預設

        if not perfect_df.empty:
            # A. 有完美個案：全部列出 (因為通過就是通過)
            list_type = "Perfect Case"
            for _, row in perfect_df.iterrows():
                best_rows_list.append({
                    'File': prefix,
                    'Type': 'Perfect',
                    'Rank': '-',
                    'Patient_ID': row[KEY_COL],
                    'Mean_SMAPE': 0.0
                })
        else:
            # B. 無完美個案：取 Top N 誤差最小者
            list_type = "Best Available"
            # 由小到大排序 (誤差越小越好)
            best_subset = smape_matrix.nsmallest(TOP_N_SHOW, 'Mean_SMAPE')
            
            rank = 1
            for _, row in best_subset.iterrows():
                best_rows_list.append({
                    'File': prefix,
                    'Type': 'Best (Non-Perfect)',
                    'Rank': rank,
                    'Patient_ID': row[KEY_COL],
                    'Mean_SMAPE': row['Mean_SMAPE']
                })
                rank += 1

        # ---------------------------------------------------------
        # 2. 產生 Worst List (Top N Worst)
        # ---------------------------------------------------------
        worst_rows_list = []
        
        # 由大到小排序 (誤差越大越差)
        worst_subset = smape_matrix.nlargest(TOP_N_SHOW, 'Mean_SMAPE')
        
        rank = 1
        for _, row in worst_subset.iterrows():
            pid = row[KEY_COL]
            mean_err = row['Mean_SMAPE']
            
            # 找出該病人誤差最大的單一特徵 (Root Cause Analysis)
            patient_errors = row[num_cols].sort_values(ascending=False)
            top_feat_name = patient_errors.index[0]
            top_feat_val = patient_errors.iloc[0]
            
            worst_rows_list.append({
                'File': prefix,
                'Rank': rank,
                'Patient_ID': pid,
                'Mean_SMAPE': mean_err,
                'Worst_Feature': f"{top_feat_name} ({top_feat_val:.1%})" # 標示出錯最大的特徵
            })
            rank += 1
        
        # ---------------------------------------------------------
        # 3. 統計摘要
        # ---------------------------------------------------------
        acceptable_count = ((smape_matrix['Mean_SMAPE'] >= 1e-9) & (smape_matrix['Mean_SMAPE'] < 0.01)).sum()
        
        stats = {
            "File": prefix,
            "Total_Patients": len(df),
            "Perfect_Count": len(perfect_df),
            "Acceptable_Count": acceptable_count,
            "List_Status": list_type
        }

        return stats, worst_rows_list, best_rows_list, smape_matrix

    except Exception as e:
        print(f"❌ {prefix} 錯誤: {e}")
        return None, [], [], None


In [31]:
# 3. 主程式
# ==========================================
all_stats = []
all_worst_list = []
all_best_list = []
matrix_map = {}

print(f"🚀 開始分析 (Top {TOP_N_SHOW} Analysis)...")
print("-" * 65)
print(f"{'File':<10} | {'Total':<6} | {'Perfect':<8} | {'Acceptable':<10} | {'List Type'}")
print("-" * 65)

for prefix in TARGET_FILES:
    stats, worst_rows, best_rows, matrix = analyze_file(BASE_DIR, prefix)
    
    if stats:
        print(f"{prefix:<10} | {stats['Total_Patients']:<6} | {stats['Perfect_Count']:<8} | {stats['Acceptable_Count']:<10} | {stats['List_Status']}")
        
        all_stats.append(stats)
        if worst_rows: all_worst_list.extend(worst_rows)
        if best_rows: all_best_list.extend(best_rows)
        matrix_map[prefix] = matrix

if all_stats:
    df_summary = pd.DataFrame(all_stats)
    df_worst = pd.DataFrame(all_worst_list)
    df_best = pd.DataFrame(all_best_list)
    
    # 格式化顯示 function
    def format_percent(x):
        if isinstance(x, (int, float)):
            if x == 0: return "0 (Perfect)"
            return f"{x:.4%}"
        return x

    if not df_worst.empty:
        df_worst['Mean_SMAPE'] = df_worst['Mean_SMAPE'].apply(format_percent)
    
    if not df_best.empty:
        df_best['Mean_SMAPE'] = df_best['Mean_SMAPE'].apply(format_percent)

    # 寫入 Excel
    try:
        print(f"\n💾 正在寫入 Excel: {OUTPUT_FILE} ...")
        with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
            
            # Sheet 1: Summary
            df_summary.to_excel(writer, sheet_name='Summary', index=False)
            
            # Sheet 2: Best Analysis (Top N)
            if not df_best.empty:
                df_best.to_excel(writer, sheet_name='Best_List', index=False)
                # 調整欄寬
                ws = writer.sheets['Best_List']
                ws.column_dimensions['A'].width = 12 # File
                ws.column_dimensions['D'].width = 20 # Patient_ID
                ws.column_dimensions['E'].width = 20 # Mean_SMAPE
            
            # Sheet 3: Worst Analysis (Top N)
            if not df_worst.empty:
                df_worst.to_excel(writer, sheet_name='Worst_List', index=False)
                ws = writer.sheets['Worst_List']
                ws.column_dimensions['A'].width = 12 # File
                ws.column_dimensions['C'].width = 20 # Patient_ID
                ws.column_dimensions['D'].width = 20 # Mean_SMAPE
                ws.column_dimensions['E'].width = 30 # Worst Feature Info

            # Sheet 4+: Matrix Data
            for name, df in matrix_map.items():
                df.to_excel(writer, sheet_name=f"{name}_Matrix", index=False)
                
        print(f"✨ 完成！Best_List 與 Worst_List 均已列出前 {TOP_N_SHOW} 筆詳細資料。")
        
    except ImportError:
        print("⚠️ 錯誤：請安裝 openpyxl (pip install openpyxl)")
else:
    print("⚠️ 無有效數據產出")

🚀 開始分析 (Top 5 Analysis)...
-----------------------------------------------------------------
File       | Total  | Perfect  | Acceptable | List Type
-----------------------------------------------------------------
EEG_t      | 117    | 0        | 0          | Best Available
GSR        | 78     | 0        | 0          | Best Available
HRV        | 78     | 0        | 62         | Best Available
RES_dc     | 78     | 78       | 0          | Perfect Case

💾 正在寫入 Excel: C:\Users\kein9\OneDrive\桌面\LAB\RelaxingBC_AC\AnalyzeData\Final_Report_With_Perfect_List.xlsx ...
✨ 完成！Best_List 與 Worst_List 均已列出前 5 筆詳細資料。
